In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

sys.path.append(str(Path('..').resolve() / 'src'))

from data_processing import (
    load_reviews_json,
    build_reviews,
    build_authors,
    attach_author_key,
    drop_missing_reviews,
    filter_recent_reviews,
    pct,
    dedupe_reviews_by_content,
    build_hotels_agg,
    write_db,
)

df_raw = load_reviews_json('../data/review.json')
df_raw.head()


,ratings,title,text,author,date_stayed,offering_id,num_helpful_votes,date,id,via_mobile
0,"{'service': 5.0, 'cleanliness': 5.0, 'overall'...","“Truly is ""Jewel of the Upper Wets Side""”",Stayed in a king suite for 11 nights and yes i...,"{'username': 'Papa_Panda', 'num_cities': 22, '...",December 2012,93338,0,2012-12-17,147643103,False
1,"{'service': 5.0, 'cleanliness': 5.0, 'overall'...",“My home away from home!”,"On every visit to NYC, the Hotel Beacon is the...","{'username': 'Maureen V', 'num_reviews': 2, 'n...",December 2012,93338,0,2012-12-17,147639004,False
2,"{'service': 4.0, 'cleanliness': 5.0, 'overall'...",“Great Stay”,This is a great property in Midtown. We two di...,"{'username': 'vuguru', 'num_cities': 12, 'num_...",December 2012,1762573,0,2012-12-18,147697954,False
3,"{'service': 5.0, 'cleanliness': 5.0, 'overall'...",“Modern Convenience”,The Andaz is a nice hotel in a central locatio...,"{'username': 'Hotel-Designer', 'num_cities': 5...",August 2012,1762573,0,2012-12-17,147625723,False
4,"{'service': 4.0, 'cleanliness': 5.0, 'overall'...",“Its the best of the Andaz Brand in the US....”,I have stayed at each of the US Andaz properti...,"{'username': 'JamesE339', 'num_cities': 34, 'n...",December 2012,1762573,0,2012-12-17,147612823,False


In [3]:
df_raw = df_raw.rename(columns={'date': 'review_date'})

# Flatten Ratings column

In [4]:
flattened_review_df = build_reviews(df_raw)
flattened_review_df.head()


,title,text,author,date_stayed,hotel_id,num_helpful_votes,review_date,review_id,via_mobile,service_rating,cleanliness_rating,overall_rating,value_rating,location_rating,sleep_quality_rating,rooms_rating,check_in_service_rating,business_service_rating
0,"“Truly is ""Jewel of the Upper Wets Side""”",Stayed in a king suite for 11 nights and yes i...,"{'username': 'Papa_Panda', 'num_cities': 22, '...",December 2012,93338,0,2012-12-17,147643103,False,5.0,5.0,5.0,5.0,5.0,5.0,5.0,NaN,NaN
1,“My home away from home!”,"On every visit to NYC, the Hotel Beacon is the...","{'username': 'Maureen V', 'num_reviews': 2, 'n...",December 2012,93338,0,2012-12-17,147639004,False,5.0,5.0,5.0,5.0,5.0,5.0,5.0,NaN,NaN
2,“Great Stay”,This is a great property in Midtown. We two di...,"{'username': 'vuguru', 'num_cities': 12, 'num_...",December 2012,1762573,0,2012-12-18,147697954,False,4.0,5.0,4.0,4.0,5.0,4.0,4.0,NaN,NaN
3,“Modern Convenience”,The Andaz is a nice hotel in a central locatio...,"{'username': 'Hotel-Designer', 'num_cities': 5...",August 2012,1762573,0,2012-12-17,147625723,False,5.0,5.0,4.0,5.0,5.0,5.0,5.0,NaN,NaN
4,“Its the best of the Andaz Brand in the US....”,I have stayed at each of the US Andaz properti...,"{'username': 'JamesE339', 'num_cities': 34, 'n...",December 2012,1762573,0,2012-12-17,147612823,False,4.0,5.0,4.0,3.0,5.0,5.0,5.0,NaN,NaN


# Create authors table and add author key to table

In [5]:
author_df = build_authors(df_raw)
author_df.head()


,username,num_cities,num_helpful_votes,num_reviews,num_type_reviews,id,location,author_key
0,Papa_Panda,22.0,12.0,29.0,24.0,8C0B42FF3C0FA366A21CFD785302A032,Gold Coast,8C0B42FF3C0FA366A21CFD785302A032|Papa_Panda
1,Maureen V,2.0,NaN,2.0,NaN,E3C85CA9DBBBC77E0DB534ABE93E4713,"Sydney, New South Wales, Australia",E3C85CA9DBBBC77E0DB534ABE93E4713|Maureen V
2,vuguru,12.0,17.0,14.0,14.0,FB1032DECE1162CB3556D05F278AAFFD,Houston,FB1032DECE1162CB3556D05F278AAFFD|vuguru
3,Hotel-Designer,5.0,26.0,5.0,5.0,EC3E275EE7590694889C8C7EE0D13961,"Laguna Beach, CA",EC3E275EE7590694889C8C7EE0D13961|Hotel-Designer
4,JamesE339,34.0,65.0,104.0,49.0,BA524A238B1171206691A6CC3F28F266,"Saint Louis, Missouri",BA524A238B1171206691A6CC3F28F266|JamesE339


In [6]:
flattened_review_df = attach_author_key(flattened_review_df, author_df)
flattened_review_df.head()


,title,text,date_stayed,hotel_id,num_helpful_votes,review_date,review_id,via_mobile,service_rating,cleanliness_rating,overall_rating,value_rating,location_rating,sleep_quality_rating,rooms_rating,check_in_service_rating,business_service_rating,author_key
0,"“Truly is ""Jewel of the Upper Wets Side""”",Stayed in a king suite for 11 nights and yes i...,December 2012,93338,0,2012-12-17,147643103,False,5.0,5.0,5.0,5.0,5.0,5.0,5.0,NaN,NaN,8C0B42FF3C0FA366A21CFD785302A032|Papa_Panda
1,“My home away from home!”,"On every visit to NYC, the Hotel Beacon is the...",December 2012,93338,0,2012-12-17,147639004,False,5.0,5.0,5.0,5.0,5.0,5.0,5.0,NaN,NaN,E3C85CA9DBBBC77E0DB534ABE93E4713|Maureen V
2,“Great Stay”,This is a great property in Midtown. We two di...,December 2012,1762573,0,2012-12-18,147697954,False,4.0,5.0,4.0,4.0,5.0,4.0,4.0,NaN,NaN,FB1032DECE1162CB3556D05F278AAFFD|vuguru
3,“Modern Convenience”,The Andaz is a nice hotel in a central locatio...,August 2012,1762573,0,2012-12-17,147625723,False,5.0,5.0,4.0,5.0,5.0,5.0,5.0,NaN,NaN,EC3E275EE7590694889C8C7EE0D13961|Hotel-Designer
4,“Its the best of the Andaz Brand in the US....”,I have stayed at each of the US Andaz properti...,December 2012,1762573,0,2012-12-17,147612823,False,4.0,5.0,4.0,3.0,5.0,5.0,5.0,NaN,NaN,BA524A238B1171206691A6CC3F28F266|JamesE339


In [12]:
author_df = author_df.sort_values(by="num_reviews", ascending=False)
author_df = author_df.drop_duplicates(subset=["author_key"])

In [13]:
# Initial check of author_df

print("\nNumber of unique values per column:")
print(author_df.nunique())

print("\nDuplicates in author_df:", author_df.duplicated().sum())
print(f"\nNumber of unique rows in author_df: {len(author_df.drop_duplicates())}")


Number of unique values per column:
username             536952
num_cities              183
num_helpful_votes       499
num_reviews             378
num_type_reviews        190
id                   576689
location              77986
author_key           576919
dtype: int64

Duplicates in author_df: 0

Number of unique rows in author_df: 576919


# Clean up review table

## Remove rows with missing data

In [15]:
# Check missing values in filtered_df in percentage
missing_values = flattened_review_df.isnull().sum()
missing_percentage = (missing_values / len(flattened_review_df)) * 100
missing_percentage

title                       0.000000
text                        0.000000
date_stayed                 7.693034
hotel_id                    0.000000
num_helpful_votes           0.000000
review_date                 0.000000
review_id                   0.000000
via_mobile                  0.000000
service_rating             13.390419
cleanliness_rating         13.513689
overall_rating              0.000000
value_rating               14.212559
location_rating            24.318972
sleep_quality_rating       42.985974
rooms_rating               19.709161
check_in_service_rating    88.642337
business_service_rating    92.518562
author_key                  0.000000
dtype: float64

In [16]:
df = flattened_review_df.copy()

df["year"] = df["review_date"].dt.year

missing_by_year = (
    df
    .groupby("year")
    .apply(lambda x: x.isna().mean() * 100)
)

print("Percentage of missing values by year:")
missing_by_year.round(2)

Percentage of missing values by year:


,title,text,date_stayed,hotel_id,num_helpful_votes,review_date,review_id,via_mobile,service_rating,cleanliness_rating,overall_rating,value_rating,location_rating,sleep_quality_rating,rooms_rating,check_in_service_rating,business_service_rating,author_key
year,,,,,,,,,,,,,,,,,,
2001,0.0,0.0,100.00,0.0,0.0,0.0,0.0,0.0,100.00,100.00,0.0,100.00,100.00,100.00,100.00,100.00,100.00,0.0
2002,0.0,0.0,96.74,0.0,0.0,0.0,0.0,0.0,96.74,96.74,0.0,96.74,100.00,100.00,96.85,100.00,100.00,0.0
2003,0.0,0.0,91.99,0.0,0.0,0.0,0.0,0.0,92.05,92.05,0.0,92.07,100.00,100.00,91.99,100.00,100.00,0.0
2004,0.0,0.0,80.26,0.0,0.0,0.0,0.0,0.0,80.44,80.39,0.0,80.50,100.00,100.00,80.32,100.00,100.00,0.0
2005,0.0,0.0,21.05,0.0,0.0,0.0,0.0,0.0,27.68,27.34,0.0,27.76,100.00,100.00,27.32,100.00,100.00,0.0
2006,0.0,0.0,8.63,0.0,0.0,0.0,0.0,0.0,19.42,17.74,0.0,20.16,61.94,100.00,17.68,61.87,76.24,0.0
2007,0.0,0.0,3.62,0.0,0.0,0.0,0.0,0.0,20.00,16.63,0.0,21.18,19.85,100.00,16.66,19.84,47.56,0.0
2008,0.0,0.0,8.63,0.0,0.0,0.0,0.0,0.0,16.99,14.06,0.0,21.57,20.79,100.00,13.43,20.79,47.25,0.0
2009,0.0,0.0,9.71,0.0,0.0,0.0,0.0,0.0,15.60,15.59,0.0,16.22,15.77,100.00,15.49,93.30,95.27,0.0


Observations:
- Sleep quality rating was only introduced in 2010, therefore all reviews from 2008 and 2009 don't have sleep_quality_rating
- Check in service rating and business service rating may be dropped as it does not have much data

In [20]:
ratings_to_require = [
    "date_stayed",
    "service_rating",
    "cleanliness_rating",
    "overall_rating",
    "value_rating",
    "location_rating",
    "rooms_rating"
]

cleaned_review_df = drop_missing_reviews(flattened_review_df, ratings_to_require)

In [21]:
df = cleaned_review_df.copy()

df["year"] = df["review_date"].dt.year

missing_by_year = (
    df
    .groupby("year")
    .apply(lambda x: x.isna().mean() * 100)
)

print("Percentage of missing values by year:")
missing_by_year.round(2)

Percentage of missing values by year:


,title,text,date_stayed,hotel_id,num_helpful_votes,review_date,review_id,via_mobile,service_rating,cleanliness_rating,overall_rating,value_rating,location_rating,sleep_quality_rating,rooms_rating,check_in_service_rating,business_service_rating,author_key
year,,,,,,,,,,,,,,,,,,
2006,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.00,0.0,0.26,36.58,0.0
2007,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.00,0.0,0.36,33.65,0.0
2008,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.00,0.0,0.36,32.44,0.0
2009,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.00,0.0,92.64,94.68,0.0
2010,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8.45,0.0,99.95,99.96,0.0
2011,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,13.84,0.0,99.99,99.99,0.0
2012,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.07,0.0,100.00,100.00,0.0


## Filter to include recent years

In [22]:
# Filter data to only include reviews from the most recent 5 years

cleaned_review_df = cleaned_review_df.rename(columns={'date': 'review_date'})
latest_year = cleaned_review_df['review_date'].max().year

filtered_review_df = cleaned_review_df[cleaned_review_df['review_date'].dt.year >= latest_year - 4]
print(f"Date range: {filtered_review_df['review_date'].min()} to {filtered_review_df['review_date'].max()}")


Date range: 2008-01-01 00:00:00 to 2012-12-20 00:00:00


## Check for duplicates

In [23]:
n = len(filtered_review_df)

dup_cols = [c for c in ["title","text"] if c in filtered_review_df.columns]
if dup_cols:
    dup_cnt = filtered_review_df.duplicated(subset=dup_cols).sum()
    print("Potential duplicate content rows:", dup_cnt, pct(dup_cnt, n))

duplicate_rows = filtered_review_df[
    filtered_review_df.duplicated(subset=dup_cols, keep=False)
].sort_values(dup_cols)

print("Number of rows involved in potential duplicates:", len(duplicate_rows))
duplicate_rows.head(10)

Potential duplicate content rows: 63 0.01%
Number of rows involved in potential duplicates: 125


,title,text,date_stayed,hotel_id,num_helpful_votes,review_date,review_id,via_mobile,service_rating,cleanliness_rating,overall_rating,value_rating,location_rating,sleep_quality_rating,rooms_rating,check_in_service_rating,business_service_rating,author_key
177746,“5 star service - comfy rooms!”,"Room service was great, so was the entire bell...",April 2011,1646128,0,2011-04-15,104169141,False,5.0,5.0,5.0,5.0,5.0,NaN,5.0,NaN,NaN,4D2533FE80EA1B398A35330AADAD9F17|ScentCurious
177747,“5 star service - comfy rooms!”,"Room service was great, so was the entire bell...",April 2011,1646128,0,2011-04-15,104146507,False,5.0,5.0,5.0,5.0,5.0,NaN,5.0,NaN,NaN,4D2533FE80EA1B398A35330AADAD9F17|ScentCurious
156527,“A European hotel in an American city”,"If you're European, then you will find this ho...",September 2012,122005,0,2012-09-28,141520666,False,3.0,4.0,3.0,4.0,4.0,3.0,3.0,NaN,NaN,|
156703,“A European hotel in an American city”,"If you're European, then you will find this ho...",September 2012,122005,0,2012-09-25,141259121,False,3.0,4.0,3.0,3.0,4.0,4.0,3.0,NaN,NaN,38EC3C73EDEF244AF604EA2F3F567FCD|Dog_Solitude
534460,"“AWSOME staff, NICE hotel!...”",I've stayed in this hotel about three times al...,February 2012,109452,0,2012-05-21,130377272,True,5.0,5.0,5.0,5.0,5.0,NaN,5.0,NaN,NaN,AEBA227E6272B9273658974838877584|y2knowledge
534536,"“AWSOME staff, NICE hotel!...”",I've stayed in this hotel about three times al...,February 2012,109452,0,2012-05-21,130373943,True,5.0,5.0,5.0,5.0,5.0,NaN,5.0,NaN,NaN,AEBA227E6272B9273658974838877584|y2knowledge
695428,“Awesome Hotel”,After a long day on the road this was the perf...,November 2011,1409115,1,2011-11-01,120028598,False,5.0,5.0,5.0,5.0,5.0,5.0,5.0,NaN,NaN,B2D8508D72D259A1A76133C87AEDF2F5|Dana K
697850,“Awesome Hotel”,After a long day on the road this was the perf...,April 2009,1409115,1,2009-04-14,28040444,False,5.0,5.0,5.0,5.0,5.0,NaN,5.0,NaN,NaN,836EA6851C1DCB13B0A2307AE00B7166|pandc123
454278,“Awesomeness”,Ashley was amazing at service at the front des...,July 2011,258655,0,2011-08-28,117340462,True,4.0,5.0,5.0,4.0,5.0,NaN,5.0,NaN,NaN,A579881C7B0584109ABC1F0F139EA3C2|noknife4u
454304,“Awesomeness”,Ashley was amazing at service at the front des...,July 2011,258655,0,2011-08-28,117337091,True,4.0,5.0,5.0,4.0,5.0,NaN,5.0,NaN,NaN,A579881C7B0584109ABC1F0F139EA3C2|noknife4u


In [24]:
# 2) Sort so the earliest date comes first
# (NaT last so missing dates don’t accidentally get kept as "earliest")
review_df, sorterd_filtered_review_df = dedupe_reviews_by_content(
    filtered_review_df,
    subset=['hotel_id', 'title', 'text'],
    date_col='review_date'
)

print('Before:', sorterd_filtered_review_df.shape)
print('After :', review_df.shape)
print('Removed:', len(filtered_review_df) - len(review_df))


Before: (598442, 18)
After : (598392, 18)
Removed: 50


# Create hotel table

In [26]:
review_cols = ["service_rating", "cleanliness_rating", "overall_rating", "value_rating", "location_rating", "sleep_quality_rating", "rooms_rating", "check_in_service_rating", "business_service_rating"]

hotels_df = build_hotels_agg(review_df, review_cols)

hotels_df


,hotel_id,num_reviews,avg_service_rating,avg_cleanliness_rating,avg_overall_rating,avg_value_rating,avg_location_rating,avg_sleep_quality_rating,avg_rooms_rating,avg_check_in_service_rating,avg_business_service_rating
0,214197,2624,2.399390,2.342226,2.363567,2.737805,4.492759,2.682187,2.131479,2.434599,2.344156
1,122005,2395,3.908559,4.206681,3.983299,3.886013,4.691441,4.117004,3.720251,3.673077,3.768116
2,93618,2300,4.015652,4.141304,3.899565,3.456957,4.634783,4.149271,3.744783,4.163842,3.580000
3,93520,2168,3.518450,3.622694,3.588561,3.588100,4.696033,3.838451,3.499077,3.025000,2.697248
4,93562,2044,4.235323,4.213307,4.088063,4.014677,4.684932,3.942877,4.002935,4.350394,3.758389
...,...,...,...,...,...,...,...,...,...,...,...
3848,277892,1,1.000000,4.000000,1.000000,1.000000,2.000000,NaN,2.000000,NaN,NaN
3849,288742,1,5.000000,5.000000,5.000000,5.000000,5.000000,NaN,5.000000,NaN,NaN
3850,1488816,1,3.000000,4.000000,3.000000,3.000000,2.000000,3.000000,3.000000,NaN,NaN
3851,1889985,1,5.000000,4.000000,5.000000,5.000000,5.000000,4.000000,5.000000,NaN,NaN


# Final checks before transforming into SQLite

In [27]:
n = len(review_df)
print("REVIEWS shape:", review_df.shape)

# PK: review_id
assert review_df["review_id"].notna().all(), "Null review_id in reviews"
assert review_df["review_id"].is_unique, "Duplicate review_id in reviews"
print("✅ review_id non-null & unique")

# FK: hotel_id
assert review_df["hotel_id"].notna().all(), "Null hotel_id in reviews"
print("✅ hotel_id non-null")

# Dates
review_df["review_date"] = pd.to_datetime(review_df["review_date"], errors="coerce")
null_dates = review_df["review_date"].isna().sum()
print("review_date null:", null_dates, pct(null_dates, n))

# Ratings range checks (only for columns that exist)
rating_cols = [c for c in review_df.columns if c.endswith("_rating")]
for c in rating_cols:
    bad = review_df[c].notna() & ~review_df[c].between(1, 5)
    if bad.any():
        print(f"⚠️ {c} out-of-range:", bad.sum(), pct(bad.sum(), n))
    else:
        pass
print("✅ ratings range check complete")

# Duplicate content (optional)
dup_cols = [c for c in ["hotel_id","review_date","title","text"] if c in review_df.columns]
if dup_cols:
    dup_cnt = review_df.duplicated(subset=dup_cols).sum()
    print("Potential duplicate content rows:", dup_cnt, pct(dup_cnt, n))


REVIEWS shape: (598392, 18)
✅ review_id non-null & unique
✅ hotel_id non-null
review_date null: 0 0.00%
✅ ratings range check complete
Potential duplicate content rows: 0 0.00%


In [28]:
nH = len(hotels_df)
print("\nHOTELS shape:", hotels_df.shape)

assert hotels_df["hotel_id"].notna().all(), "Null hotel_id in hotels"
assert hotels_df["hotel_id"].is_unique, "Duplicate hotel_id in hotels"
print("✅ hotel_id non-null & unique")

missing_hotel_fk = set(review_df["hotel_id"].dropna()) - set(hotels_df["hotel_id"])
print("Missing hotel_id referenced by reviews:", len(missing_hotel_fk))


HOTELS shape: (3853, 11)
✅ hotel_id non-null & unique
Missing hotel_id referenced by reviews: 0


In [29]:
print("\n=== DATA FOUNDATION SUMMARY ===")
print("Total reviews:", len(review_df))
print("Unique hotels:", review_df["hotel_id"].nunique())
if "author_key" in review_df.columns:
    print("Unique authors in reviews:", review_df["author_key"].nunique())
print("Review date range:", review_df["review_date"].min(), "→", review_df["review_date"].max())
core = [c for c in ["overall_rating","service_rating","cleanliness_rating","value_rating","location_rating","rooms_rating"] if c in review_df.columns]
if core:
    kept = len(review_df.dropna(subset=core))
    print(f"Rows with all core ratings present ({len(core)} cols):", kept, pct(kept, len(review_df)))


=== DATA FOUNDATION SUMMARY ===
Total reviews: 598392
Unique hotels: 3853
Unique authors in reviews: 450193
Review date range: 2008-01-01 00:00:00 → 2012-12-20 00:00:00
Rows with all core ratings present (6 cols): 598392 100.00%


# Transform into sqlite (full data and sample)

# Check DB file

In [30]:
import sqlite3

conn = sqlite3.connect("../data/reviews_sample.db")

pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table'",
    conn
)

,name
0,reviews
1,authors
2,hotels


In [31]:
print(pd.read_sql("SELECT COUNT(*) AS n_reviews FROM reviews", conn))
print(pd.read_sql("SELECT COUNT(*) AS n_authors FROM authors", conn))
print(pd.read_sql("SELECT COUNT(*) AS n_hotels  FROM hotels",  conn))

   n_reviews
0       6000
   n_authors
0       5929
   n_hotels
0      1872


In [32]:
print(pd.read_sql("PRAGMA table_info(reviews)", conn))
print(pd.read_sql("PRAGMA table_info(authors)", conn))
print(pd.read_sql("PRAGMA table_info(hotels)", conn))

    cid                     name     type  notnull dflt_value  pk
0     0                    title     TEXT        0       None   0
1     1                     text     TEXT        0       None   0
2     2              date_stayed     TEXT        0       None   0
3     3                 hotel_id  INTEGER        0       None   0
4     4        num_helpful_votes  INTEGER        0       None   0
5     5              review_date     TEXT        0       None   0
6     6                review_id  INTEGER        0       None   0
7     7               via_mobile  INTEGER        0       None   0
8     8           service_rating     REAL        0       None   0
9     9       cleanliness_rating     REAL        0       None   0
10   10           overall_rating     REAL        0       None   0
11   11             value_rating     REAL        0       None   0
12   12          location_rating     REAL        0       None   0
13   13     sleep_quality_rating     REAL        0       None   0
14   14   

In [33]:
pd.read_sql(
    "SELECT name, tbl_name FROM sqlite_master WHERE type='index'",
    conn
)

,name,tbl_name
0,idx_reviews_hotel,reviews
1,idx_reviews_date,reviews
2,idx_reviews_hotel_date,reviews
3,ux_reviews_review_id,reviews
4,ux_hotels_hotel_id,hotels
5,idx_authors_author_key,authors
6,ux_authors_author_key,authors
7,idx_hotels_hotel_id,hotels


In [34]:
pd.read_sql("""
SELECT COUNT(*) AS dup_review_ids
FROM (
    SELECT review_id
    FROM reviews
    GROUP BY review_id
    HAVING COUNT(*) > 1
)
""", conn)


,dup_review_ids
0,0


In [35]:
pd.read_sql("""
SELECT COUNT(*) AS orphan_reviews
FROM reviews r
LEFT JOIN hotels h ON r.hotel_id = h.hotel_id
WHERE h.hotel_id IS NULL
""", conn)


,orphan_reviews
0,0


In [36]:
pd.read_sql("""
SELECT review_id, hotel_id, review_date, overall_rating
FROM reviews
LIMIT 10
""", conn)


,review_id,hotel_id,review_date,overall_rating
0,124253920,84107,2012-02-07,4.0
1,129859003,122015,2012-05-13,4.0
2,139816807,80311,2012-09-09,3.0
3,126236988,1783324,2012-03-17,3.0
4,118379450,119601,2011-09-20,5.0
5,30314068,478253,2009-05-19,5.0
6,144423432,563989,2012-11-03,4.0
7,23162137,95302,2009-01-01,2.0
8,147426180,242394,2012-12-14,4.0
9,120253620,100507,2011-11-06,5.0


In [37]:
pd.read_sql("""
EXPLAIN QUERY PLAN
SELECT AVG(overall_rating)
FROM reviews
WHERE hotel_id = (
    SELECT hotel_id FROM reviews LIMIT 1
)
AND review_date >= '2011-01-01'
""", conn)


,id,parent,notused,detail
0,4,0,0,SEARCH reviews USING INDEX idx_reviews_hotel_d...
1,7,0,0,SCALAR SUBQUERY 1
2,15,7,0,SCAN reviews USING COVERING INDEX idx_reviews_...


In [38]:
conn.close()